# Understanding Differentiable Hierarchical Tokenization (dHT)

This notebook provides a comprehensive introduction to the **Differentiable Hierarchical Tokenization (dHT)** approach and demonstrates how to use it for visual tokenization.

## What is dHT?

dHT is a novel approach to visual tokenization that:
- **Adapts to image content** with pixel-level granularity
- **Hierarchically segments** images into meaningful regions
- **Is fully differentiable** and can be trained end-to-end
- **Is backward compatible** with existing Vision Transformer architectures

Unlike fixed patch-based tokenization (e.g., 16x16 patches in standard ViT), dHT creates tokens that:
1. Vary in size and shape
2. Align with semantic boundaries in images
3. Efficiently represent both simple and complex regions

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Import dHT modules
from dht.tok.tokenizer import dHTTokenizer

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Basic Tokenization Example

Let's start by creating a simple image and seeing how dHT tokenizes it.

In [ ]:
# Create a simple synthetic image with distinct regions
def create_sample_image(size=224):
    """Create a simple test image with distinct color regions."""
    img = torch.zeros(1, 3, size, size)
    
    # Create different colored regions
    img[:, 0, :size//2, :size//2] = 0.8  # Red region (top-left)
    img[:, 1, :size//2, size//2:] = 0.8  # Green region (top-right)
    img[:, 2, size//2:, :size//2] = 0.8  # Blue region (bottom-left)
    img[:, :, size//2:, size//2:] = 0.6  # Gray region (bottom-right)
    
    # Add a small detailed region
    img[:, 0, size//4:size//4+20, size//4:size//4+20] = 1.0
    img[:, 1, size//4:size//4+20, size//4:size//4+20] = 0.5
    
    return img

# Create sample image
sample_img = create_sample_image(224).to(device)

# Visualize
plt.figure(figsize=(6, 6))
plt.imshow(sample_img[0].permute(1, 2, 0).cpu().numpy())
plt.title("Sample Input Image")
plt.axis('off')
plt.show()

## 2. Initialize and Use dHT Tokenizer

In [ ]:
# Initialize tokenizer
tokenizer = dHTTokenizer(
    in_ch=3,           # RGB channels
    hid_ch=8,          # Hidden feature channels
    similarity='gaussian',  # Similarity metric
    criterion='aicc',  # Information criterion
    iota=5.0,         # Information criterion weight
    cmp=0.1,          # Compression parameter
    compute_grad=True, # Compute gradient features
).to(device)

print("Tokenizer initialized")

In [ ]:
# Tokenize the image
with torch.no_grad():
    tokenizer.eval()
    result = tokenizer(sample_img)

print(f"Number of tokens: {result.nV}")
print(f"Token features shape: {result.fV.shape}")
print(f"Segmentation shape: {result.seg.shape}")

## 3. Visualize Results

In [ ]:
# Visualize segmentation
def visualize_segmentation(seg, original_img=None):
    seg_np = seg[0].cpu().numpy()
    n_segments = seg_np.max() + 1
    colors = plt.cm.tab20(np.linspace(0, 1, n_segments))
    colored_seg = colors[seg_np]
    
    fig, axes = plt.subplots(1, 2 if original_img is not None else 1, figsize=(12, 6))
    
    if original_img is not None:
        axes[0].imshow(original_img[0].permute(1, 2, 0).cpu().numpy())
        axes[0].set_title("Original Image")
        axes[0].axis('off')
        
        axes[1].imshow(colored_seg[:, :, :3])
        axes[1].set_title(f"dHT Segmentation ({n_segments} tokens)")
        axes[1].axis('off')
    else:
        axes.imshow(colored_seg[:, :, :3])
        axes.set_title(f"dHT Segmentation ({n_segments} tokens)")
        axes.axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_segmentation(result.seg, sample_img)